# Фінальний проєкт з реляційних баз даних

**Герасимюк Юлія**

У цьому проєкті я працюю з датасетом `infectious_cases`, який містить дані про інфекційні захворювання в різних країнах за роками.

Спочатку завантажу дані в PostgreSQL, потім нормалізую таблицю та виконаю SQL-запити за завданням. Також створю дві функції для роботи з датами та кількістю захворювань.

## 1. Підготовка PostgreSQL

Спочатку встановлю потрібні бібліотеки, щоб працювати з PostgreSQL прямо в Google Colab. Так мені не потрібно окремо встановлювати базу даних на комп'ютер.

In [65]:

# Встановлюю бібліотеки для роботи з PostgreSQL
!pip -q install pgserver psycopg2-binary sqlalchemy pandas


In [66]:

import pgserver
import psycopg2

# Вказую папку для даних PostgreSQL
pgdata = "/tmp/pgdata_final_project"

# Запускаю сервер PostgreSQL
server = pgserver.get_server(pgdata)
server.ensure_postgres_running()

# Підключаюся до бази даних
conn = psycopg2.connect(server.get_uri())
conn.autocommit = True

# Перевіряю, чи працює PostgreSQL
with conn.cursor() as cur:
    cur.execute("SELECT version();")
    print("PostgreSQL працює!")
    print(cur.fetchone()[0])


PostgreSQL працює!
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 2. Створення схеми pandemic

Спочатку створю окрему схему `pandemic`, у якій будуть зберігатися таблиці мого проєкту. Так усі дані будуть в одному місці.

In [67]:
# Створюю схему pandemic
with conn.cursor() as cur:
    cur.execute("CREATE SCHEMA IF NOT EXISTS pandemic;")
    cur.execute("SET search_path TO pandemic;")

    # Перевіряю активну схему
    cur.execute("SELECT current_schema();")
    print("Активна схема:", cur.fetchone()[0])

Активна схема: pandemic


## 3. Завантаження даних

Завантажу CSV-файл із Google Drive за допомогою pandas. Спочатку перегляну перші рядки, назви стовпців і розмір датасету, щоб зрозуміти його структуру перед створенням таблиці в PostgreSQL.

In [68]:
import pandas as pd
import requests
from io import BytesIO

# Завантажую CSV-файл із Google Drive
file_id = "1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

response = requests.get(url, timeout=60)
response.raise_for_status()

# Зчитую дані
df = pd.read_csv(BytesIO(response.content))

# Переглядаю перші 5 рядків
display(df.head())

# Перевіряю розмір і назви стовпців
print("Кількість рядків:", df.shape[0])
print("Кількість стовпців:", df.shape[1])
print("Назви стовпців:", df.columns.tolist())

,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


Кількість рядків: 10521
Кількість стовпців: 12
Назви стовпців: ['Entity', 'Code', 'Year', 'Number_yaws', 'polio_cases', 'cases_guinea_worm', 'Number_rabies', 'Number_malaria', 'Number_hiv', 'Number_tuberculosis', 'Number_smallpox', 'Number_cholera_cases']


## 4. Створення таблиці infectious_cases

Створю таблицю infectious_cases у схемі pandemic. Назви стовпців залишу такими, як у CSV-файлі, щоб було зручно завантажувати дані.

Для назв країн використаю TEXT, для року - INTEGER, а для кількості захворювань - NUMERIC, оскільки в даних є числові значення та пропуски.

In [69]:
# Створюю початкову таблицю для даних із CSV
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS pandemic.infectious_cases (
            "Entity" TEXT,
            "Code" TEXT,
            "Year" INTEGER,
            "Number_yaws" NUMERIC,
            "polio_cases" NUMERIC,
            "cases_guinea_worm" NUMERIC,
            "Number_rabies" NUMERIC,
            "Number_malaria" NUMERIC,
            "Number_hiv" NUMERIC,
            "Number_tuberculosis" NUMERIC,
            "Number_smallpox" NUMERIC,
            "Number_cholera_cases" NUMERIC
        );
    """)

    # Перевіряю, чи створилася таблиця
    cur.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'pandemic';
    """)

    print("Таблиці у схемі pandemic:")
    print(cur.fetchall())

Таблиці у схемі pandemic:
[('entities',), ('infectious_cases',), ('infectious_cases_normalized',)]


## 5. Завантаження даних у таблицю

Тепер перенесу дані з CSV-файлу в таблицю PostgreSQL. Пропущені значення збережу як NULL.

Після завантаження перевірю кількість рядків і перегляну перші записи, щоб переконатися, що дані перенеслися правильно.

In [70]:
from io import StringIO

# Готую дані для завантаження в PostgreSQL
csv_buffer = StringIO()
df.to_csv(csv_buffer, index=False, header=False, na_rep="")
csv_buffer.seek(0)

# Завантажую дані в таблицю
with conn.cursor() as cur:
    # Очищую таблицю перед завантаженням, щоб не було дублікатів
    cur.execute("TRUNCATE TABLE pandemic.infectious_cases;")

    cur.copy_expert("""
        COPY pandemic.infectious_cases
        FROM STDIN
        WITH (FORMAT CSV, NULL '')
    """, csv_buffer)

    # Перевіряю кількість завантажених рядків
    cur.execute("SELECT COUNT(*) FROM pandemic.infectious_cases;")
    print("Кількість рядків у PostgreSQL:", cur.fetchone()[0])

    # Переглядаю перші 5 записів
    cur.execute("""
        SELECT *
        FROM pandemic.infectious_cases
        LIMIT 5;
    """)

    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

display(pd.DataFrame(rows, columns=columns))

Кількість рядків у PostgreSQL: 10521


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,None,6160.0,0.0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859.0,0.0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730.0,0.0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937.0,0.0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864.0,0.0,None,None,None,None,None,None


## 6. Нормалізація даних до 3НФ

У початковій таблиці назви та коди країн повторюються для кожного року. Щоб не зберігати однакові дані багато разів, розділю їх на дві таблиці.

Спочатку створю таблицю entities з унікальними парами Entity та Code. Для кожної пари буде окремий ID, який потім використаю для зв'язку з таблицею захворювань.

In [71]:
# Створюю таблицю з унікальними країнами та кодами
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS pandemic.entities (
            id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            "Entity" TEXT,
            "Code" TEXT
        );
    """)

    # Додаю тільки ті пари, яких ще немає в таблиці
    cur.execute("""
        INSERT INTO pandemic.entities ("Entity", "Code")
        SELECT DISTINCT i."Entity", i."Code"
        FROM pandemic.infectious_cases AS i
        WHERE NOT EXISTS (
            SELECT 1
            FROM pandemic.entities AS e
            WHERE e."Entity" IS NOT DISTINCT FROM i."Entity"
              AND e."Code" IS NOT DISTINCT FROM i."Code"
        );
    """)

    # Перевіряю кількість записів
    cur.execute("SELECT COUNT(*) FROM pandemic.entities;")
    print("Кількість унікальних пар:", cur.fetchone()[0])

    # Переглядаю перші 10 записів
    cur.execute("""
        SELECT *
        FROM pandemic.entities
        ORDER BY id
        LIMIT 10;
    """)

    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

display(pd.DataFrame(rows, columns=columns))

Кількість унікальних пар: 245


,id,Entity,Code
0,1,Russia,RUS
1,2,Trinidad and Tobago,TTO
2,3,Antigua and Barbuda,ATG
3,4,Nicaragua,NIC
4,5,Denmark,DNK
5,6,Turkey,TUR
6,7,Sudan,SDN
7,8,Myanmar,MMR
8,9,Macao,MAC
9,10,Togo,TGO


### Створення таблиці з даними про захворювання

Тепер створю другу таблицю, у якій зберігатимуться роки та показники захворювань.

Замість назв і кодів країн використаю entity_id. Він буде зовнішнім ключем, який зв'язує цю таблицю з entities.

In [72]:
# Створюю таблицю з даними про захворювання
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS pandemic.infectious_cases_normalized (
            id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            entity_id INTEGER REFERENCES pandemic.entities(id),
            "Year" INTEGER,
            "Number_yaws" NUMERIC,
            "polio_cases" NUMERIC,
            "cases_guinea_worm" NUMERIC,
            "Number_rabies" NUMERIC,
            "Number_malaria" NUMERIC,
            "Number_hiv" NUMERIC,
            "Number_tuberculosis" NUMERIC,
            "Number_smallpox" NUMERIC,
            "Number_cholera_cases" NUMERIC
        );
    """)

    # Перевіряю, чи створилася таблиця
    cur.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'pandemic'
        ORDER BY table_name;
    """)

    print("Таблиці у схемі pandemic:")
    for row in cur.fetchall():
        print(row[0])

Таблиці у схемі pandemic:
entities
infectious_cases
infectious_cases_normalized


### Перенесення даних у нормалізовану таблицю

Тепер перенесу всі записи з початкової таблиці в infectious_cases_normalized.

За допомогою JOIN знайду ID для кожної країни з таблиці entities. Після перенесення перевірю кількість рядків і перегляну дані, щоб переконатися, що зв'язок працює правильно.

In [73]:
# Переношу дані в нормалізовану таблицю
with conn.cursor() as cur:

    # Очищую таблицю, щоб уникнути дублікатів
    cur.execute("""
        TRUNCATE TABLE pandemic.infectious_cases_normalized
        RESTART IDENTITY;
    """)

    # Додаю записи, замінюючи Entity і Code на entity_id
    cur.execute("""
        INSERT INTO pandemic.infectious_cases_normalized (
            entity_id, "Year", "Number_yaws", "polio_cases",
            "cases_guinea_worm", "Number_rabies", "Number_malaria",
            "Number_hiv", "Number_tuberculosis",
            "Number_smallpox", "Number_cholera_cases"
        )
        SELECT
            e.id, i."Year", i."Number_yaws", i."polio_cases",
            i."cases_guinea_worm", i."Number_rabies", i."Number_malaria",
            i."Number_hiv", i."Number_tuberculosis",
            i."Number_smallpox", i."Number_cholera_cases"
        FROM pandemic.infectious_cases AS i
        JOIN pandemic.entities AS e
            ON i."Entity" IS NOT DISTINCT FROM e."Entity"
           AND i."Code" IS NOT DISTINCT FROM e."Code";
    """)

    # Перевіряю кількість перенесених записів
    cur.execute("""
        SELECT COUNT(*)
        FROM pandemic.infectious_cases_normalized;
    """)
    print("Кількість рядків:", cur.fetchone()[0])

    # Перевіряю зв'язок між таблицями
    cur.execute("""
        SELECT e."Entity", e."Code", n."Year", n."Number_rabies"
        FROM pandemic.infectious_cases_normalized AS n
        JOIN pandemic.entities AS e ON n.entity_id = e.id
        ORDER BY e."Entity", n."Year"
        LIMIT 10;
    """)

    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

display(pd.DataFrame(rows, columns=columns))

Кількість рядків: 10521


,Entity,Code,Year,Number_rabies
0,Afghanistan,AFG,1960,None
1,Afghanistan,AFG,1965,None
2,Afghanistan,AFG,1980,None
3,Afghanistan,AFG,1981,None
4,Afghanistan,AFG,1982,None
5,Afghanistan,AFG,1983,None
6,Afghanistan,AFG,1984,None
7,Afghanistan,AFG,1985,None
8,Afghanistan,AFG,1986,None
9,Afghanistan,AFG,1987,None


## 7. Аналіз даних про сказ

Тепер проаналізую дані про захворюваність на сказ для кожної країни, території або групи, яка є в датасеті.

За допомогою GROUP BY та агрегатних функцій AVG, MIN, MAX і SUM знайду основні показники. Пропущені значення NULL не враховуватиму в розрахунках.

Результати відсортую за середньою кількістю випадків від найбільшої до найменшої та виведу перші 10 записів.

In [74]:
# Розраховую статистику захворюваності на сказ
query = """
    SELECT
        e."Entity",
        e."Code",
        ROUND(AVG(n."Number_rabies"), 2) AS avg_rabies,
        MIN(n."Number_rabies") AS min_rabies,
        MAX(n."Number_rabies") AS max_rabies,
        SUM(n."Number_rabies") AS sum_rabies
    FROM pandemic.infectious_cases_normalized AS n
    JOIN pandemic.entities AS e
        ON n.entity_id = e.id
    WHERE n."Number_rabies" IS NOT NULL
    GROUP BY e.id, e."Entity", e."Code"
    ORDER BY avg_rabies DESC
    LIMIT 10;
"""

# Виконую SQL-запит і показую результат
with conn.cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

display(pd.DataFrame(rows, columns=columns))

,Entity,Code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,World,OWID_WRL,20192.37,14075.508,24744.658,605771.111
1,Lower Middle Income (WB),None,15193.96,10202.528,19182.795,455818.7815
2,South Asia (WB),None,11729.89,7271.2754,15361.878,351896.6626
3,South-East Asia Region (WHO),None,11424.33,6806.0127,15641.958,342729.8068
4,G20,None,10189.05,6339.0796,13164.881,305671.3801
5,India,IND,8599.17,5425.8726,11121.142,257975.1984
6,Sub-Saharan Africa (WB),None,4748.27,4356.8184,5017.977,142448.0427
7,African Region (WHO),None,4649.70,4283.073,4917.784,139491.0713
8,Low Income (WB),None,3568.11,3014.617,3873.533,107043.1635
9,East Asia & Pacific (WB),None,3427.46,2009.9395,4591.0107,102823.8340


## 8. Робота з датами

У таблиці рік зберігається як число. Для роботи з датами створю дату 1 січня відповідного року за допомогою MAKE_DATE().

Також використаю CURRENT_DATE для отримання сьогоднішньої дати та AGE() і EXTRACT(), щоб порахувати різницю в роках.

In [75]:
# Створюю дати та розраховую різницю в роках
query = """
    SELECT
        id,
        entity_id,
        "Year",
        MAKE_DATE("Year", 1, 1) AS year_date,
        CURRENT_DATE AS today_date,
        EXTRACT(
            YEAR FROM AGE(CURRENT_DATE, MAKE_DATE("Year", 1, 1))
        ) AS years_difference
    FROM pandemic.infectious_cases_normalized
    ORDER BY id
    LIMIT 10;
"""

# Виконую запит і переглядаю результат
with conn.cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

display(pd.DataFrame(rows, columns=columns))

,id,entity_id,Year,year_date,today_date,years_difference
0,1,212,1980,1980-01-01,2026-10-14,46
1,2,212,1981,1981-01-01,2026-10-14,45
2,3,212,1982,1982-01-01,2026-10-14,44
3,4,212,1983,1983-01-01,2026-10-14,43
4,5,212,1984,1984-01-01,2026-10-14,42
5,6,212,1985,1985-01-01,2026-10-14,41
6,7,212,1986,1986-01-01,2026-10-14,40
7,8,212,1987,1987-01-01,2026-10-14,39
8,9,212,1988,1988-01-01,2026-10-14,38
9,10,212,1989,1989-01-01,2026-10-14,37


## 9. Створення SQL-функцій

### Функція для розрахунку різниці в роках

Спочатку створю функцію, яка приймає рік і повертає різницю між ним та поточною датою.

Для цього використаю MAKE_DATE(), CURRENT_DATE, AGE() та EXTRACT(). Так я зможу повторно використовувати цей розрахунок у SQL-запитах.

In [76]:
# Створюю функцію для розрахунку різниці в роках
with conn.cursor() as cur:
    cur.execute("""
        CREATE OR REPLACE FUNCTION pandemic.years_from_year(input_year INTEGER)
        RETURNS INTEGER
        LANGUAGE SQL
        AS $$
            SELECT EXTRACT(
                YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(input_year, 1, 1))
            )::INTEGER;
        $$;
    """)

    # Перевіряю роботу функції
    cur.execute("""
        SELECT
            "Year",
            pandemic.years_from_year("Year") AS years_difference
        FROM pandemic.infectious_cases_normalized
        ORDER BY id
        LIMIT 10;
    """)

    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

display(pd.DataFrame(rows, columns=columns))

,Year,years_difference
0,1980,46
1,1981,45
2,1982,44
3,1983,43
4,1984,42
5,1985,41
6,1986,40
7,1987,39
8,1988,38
9,1989,37


### Функція для розрахунку середньої кількості захворювань

Створю ще одну функцію, яка приймає кількість випадків захворювання за рік і число, на яке потрібно її поділити.

Перевірю роботу функції на даних про сказ. Розрахую середню кількість випадків за місяць, квартал і півріччя.

Для уникнення ділення на нуль використаю NULLIF().

In [77]:
# Створюю функцію для розрахунку середньої кількості випадків
with conn.cursor() as cur:
    cur.execute("""
        CREATE OR REPLACE FUNCTION pandemic.average_cases(
            cases NUMERIC,
            periods INTEGER
        )
        RETURNS NUMERIC
        LANGUAGE SQL
        AS $$
            SELECT cases / NULLIF(periods, 0);
        $$;
    """)

    # Перевіряю функцію для місяця, кварталу та півріччя
    cur.execute("""
        SELECT
            e."Entity",
            n."Year",
            n."Number_rabies",
            ROUND(pandemic.average_cases(n."Number_rabies", 12), 2)
                AS avg_per_month,
            ROUND(pandemic.average_cases(n."Number_rabies", 4), 2)
                AS avg_per_quarter,
            ROUND(pandemic.average_cases(n."Number_rabies", 2), 2)
                AS avg_per_half_year
        FROM pandemic.infectious_cases_normalized AS n
        JOIN pandemic.entities AS e
            ON n.entity_id = e.id
        WHERE n."Number_rabies" IS NOT NULL
        ORDER BY e."Entity", n."Year"
        LIMIT 10;
    """)

    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

display(pd.DataFrame(rows, columns=columns))

,Entity,Year,Number_rabies,avg_per_month,avg_per_quarter,avg_per_half_year
0,Afghanistan,1990,2.145772,0.18,0.54,1.07
1,Afghanistan,1991,2.3209748,0.19,0.58,1.16
2,Afghanistan,1992,2.6180408,0.22,0.65,1.31
3,Afghanistan,1993,2.8531172,0.24,0.71,1.43
4,Afghanistan,1994,2.9899871,0.25,0.75,1.49
5,Afghanistan,1995,3.126361,0.26,0.78,1.56
6,Afghanistan,1996,3.281437,0.27,0.82,1.64
7,Afghanistan,1997,3.446122,0.29,0.86,1.72
8,Afghanistan,1998,3.5570247,0.30,0.89,1.78
9,Afghanistan,1999,3.607037,0.30,0.90,1.80


## 10. Перевірка результатів

На завершення перевірю кількість записів у початковій та нормалізованій таблицях, а також кількість унікальних країн і територій.

Окремо перевірю, чи всі записи мають правильний зв'язок із таблицею entities.

In [78]:
# Перевіряю результати нормалізації
query = """
    SELECT
        (SELECT COUNT(*)
         FROM pandemic.infectious_cases) AS original_rows,

        (SELECT COUNT(*)
         FROM pandemic.infectious_cases_normalized) AS normalized_rows,

        (SELECT COUNT(*)
         FROM pandemic.entities) AS unique_entities,

        (SELECT COUNT(*)
         FROM pandemic.infectious_cases_normalized AS n
         LEFT JOIN pandemic.entities AS e
             ON n.entity_id = e.id
         WHERE e.id IS NULL) AS missing_connections;
"""

with conn.cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

display(pd.DataFrame(rows, columns=columns))

,original_rows,normalized_rows,unique_entities,missing_connections
0,10521,10521,245,0


## 11. Висновок

У цьому проєкті я працювала з даними про інфекційні захворювання в PostgreSQL.

Спочатку створила схему pandemic і завантажила CSV-файл із 10 521 записом.

Потім нормалізувала дані до 3НФ, розділивши їх на дві таблиці: entities та infectious_cases_normalized. Для зв'язку між ними використала зовнішній ключ entity_id.

За допомогою SQL-запитів розрахувала середнє, мінімальне, максимальне та сумарне значення захворюваності на сказ. Також попрацювала з датами та створила дві SQL-функції для розрахунку різниці в роках і середньої кількості випадків захворювання за різні періоди.

Наприкінці перевірила результати нормалізації. Кількість записів залишилася незмінною, а всі зв'язки між таблицями працюють правильно.

Під час виконання проєкту я закріпила навички роботи з PostgreSQL, нормалізацією даних, JOIN, GROUP BY, агрегатними функціями та SQL-функціями.